# AIC Cable Insertion — ACT Policy Training

Trains an **ACT (Action Chunking Transformer)** policy on your HuggingFace dataset using a Colab GPU.  
Checkpoints are saved to **Google Drive** after every `SAVE_FREQ` steps so you can resume any time.

---

## Before you start

1. **Runtime → Change runtime type → T4 GPU** (free tier) or A100 (Colab Pro)
2. Your dataset must be uploaded to HuggingFace Hub  
   (`pixi run python my_policy_node/scripts/push_dataset_to_hub.py ...`)
3. You need a HuggingFace account (free) — login prompt appears in Step 5

---

## Quick-start

| Step | What it does |
|------|--------------|
| 1 | Check GPU |
| 2 | Mount Google Drive (for persistent checkpoints) |
| 3 | Install LeRobot |
| 4 | **Fill in your config** (HF username, dataset, etc.) |
| 5 | HuggingFace login |
| 6 | Preview dataset features |
| 7 | **Run training** |
| 8 | Resume after disconnection |
| 9 | Inspect checkpoints |

---
## Step 1 — Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to  Runtime → Change runtime type → Hardware accelerator → T4 GPU"
    )

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {name}")
print(f"VRAM : {vram:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

---
## Step 2 — Mount Google Drive

Checkpoints are written to `My Drive/aic_training/<dataset>_act/`.  
They persist across Colab sessions so you can always resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

---
## Step 3 — Install LeRobot

Pins to **v0.5.1** to match the version used for data collection.  
This cell takes ~2 minutes on first run.

In [ ]:
# Install LeRobot at the same version used for data collection
!pip install -q "lerobot==0.5.1" "huggingface_hub>=0.23" wandb

import lerobot
print(f"LeRobot {lerobot.__version__} ready")

---
## Step 4 — Configuration

**Edit the values in this cell before running anything else.**

In [ ]:
# ============================================================
#  YOUR SETTINGS — edit these
# ============================================================

HF_USERNAME   = "your_hf_username"      # @param {type:"string"}
DATASET_NAME  = "sfp_insertion_demos"   # @param {type:"string"}

# Training duration
STEPS         = 100_000   # @param {type:"integer"}   total gradient steps
BATCH_SIZE    = 8         # @param {type:"integer"}   reduce to 4 if OOM

# ACT action chunk — how many future actions to predict at once.
# 50 = 5 seconds at 10 Hz.  Good starting point for insertion tasks.
CHUNK_SIZE    = 50        # @param {type:"integer"}

# Checkpointing — Drive checkpoint every SAVE_FREQ steps
SAVE_FREQ     = 2_000     # @param {type:"integer"}

# Weights & Biases (optional — leave empty to skip)
WANDB_PROJECT = ""        # @param {type:"string"}   e.g. "aic_policy"

# ============================================================
#  Derived — do not edit
# ============================================================

DATASET_REPO_ID = f"{HF_USERNAME}/{DATASET_NAME}"
CHECKPOINT_DIR  = f"/content/drive/MyDrive/aic_training/{DATASET_NAME}_act"
USE_WANDB       = bool(WANDB_PROJECT)

print(f"Dataset      : {DATASET_REPO_ID}")
print(f"Steps        : {STEPS:,}   Batch: {BATCH_SIZE}   Chunk: {CHUNK_SIZE}")
print(f"Save every   : {SAVE_FREQ:,} steps")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"W&B          : {'enabled — project=' + WANDB_PROJECT if USE_WANDB else 'disabled'}")

---
## Step 5 — HuggingFace Login

Needed to download your (private) dataset.  
Get a token at https://huggingface.co/settings/tokens (read access is enough).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

---
## Step 6 — Preview Dataset

Shows the features present in the dataset and builds the policy input/output config.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.configs.types import FeatureType, PolicyFeature

print(f"Loading dataset metadata from {DATASET_REPO_ID} ...")
ds = LeRobotDataset(DATASET_REPO_ID)

print(f"\nEpisodes : {ds.num_episodes}")
print(f"Frames   : {len(ds):,}")
print(f"FPS      : {ds.fps}")
print(f"\nFeatures:")
for k, v in sorted(ds.features.items()):
    print(f"  {k:<50s}  shape={str(v['shape']):<15s}  dtype={v['dtype']}")

# ---------- Build PolicyFeature dicts from dataset metadata ----------
input_features  = {}
output_features = {}

for key, feat in ds.features.items():
    shape = tuple(feat["shape"])
    if key.startswith("observation.images"):
        input_features[key] = PolicyFeature(type=FeatureType.VISUAL, shape=shape)
    elif key.startswith("observation."):
        input_features[key] = PolicyFeature(type=FeatureType.STATE, shape=shape)
    elif key == "action":
        output_features[key] = PolicyFeature(type=FeatureType.ACTION, shape=shape)

print(f"\nPolicy inputs  ({len(input_features)} features):")
for k, v in input_features.items():
    print(f"  {v.type.value:<8s}  {k}  {v.shape}")
print(f"\nPolicy outputs ({len(output_features)} features):")
for k, v in output_features.items():
    print(f"  {v.type.value:<8s}  {k}  {v.shape}")

---
## Step 7 — Train

- If `CHECKPOINT_DIR` already contains a checkpoint from a previous run, training **resumes automatically**.
- Checkpoints are saved to Drive every `SAVE_FREQ` steps.
- Expected time on a T4 GPU: ~2–3 hours for 100 k steps.

> **Tip:** If the session disconnects, just re-run Steps 1–6 and then this cell — it will pick up where it left off.

In [ ]:
import sys
import shutil
from pathlib import Path

# ── Patch: inject missing 'names' for image features ─────────────────────
# LeRobot v0.5.1's dataset_to_policy_features requires a 'names' key on
# every 3-D feature (images).  Our dataset was collected without it.
# factory.py imports the function directly, so patch both namespaces.
import lerobot.datasets.feature_utils as _fu
import lerobot.policies.factory as _pf

_orig_d2pf = _fu.dataset_to_policy_features

def _d2pf_no_names(features):
    fixed = {}
    for k, v in features.items():
        v = dict(v)
        if len(v.get("shape", [])) == 3 and "names" not in v:
            v["names"] = ["channel", "height", "width"]  # our images are CHW
        fixed[k] = v
    return _orig_d2pf(fixed)

_fu.dataset_to_policy_features = _d2pf_no_names
_pf.dataset_to_policy_features = _d2pf_no_names  # factory imports it directly

# ── Detect whether to resume ─────────────────────────────────────────────
from lerobot.configs.default import DatasetConfig
from lerobot.configs.train import TrainPipelineConfig
from lerobot.configs.types import NormalizationMode
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.scripts.lerobot_train import train

ckpt_root         = Path(CHECKPOINT_DIR)
train_config_json = ckpt_root / "train_config.json"

if ckpt_root.is_dir() and not train_config_json.is_file():
    shutil.rmtree(ckpt_root)
    print("Removed stale output directory (no train_config.json) — starting fresh")

IS_RESUME = train_config_json.is_file()

if IS_RESUME:
    checkpoints = sorted(ckpt_root.glob("checkpoints/*/"))
    if checkpoints:
        print(f"Resuming from checkpoint: {checkpoints[-1].name}")
    else:
        print("Resuming — config found but no checkpoint yet (will start from step 0)")
    sys.argv = [sys.argv[0], f"--config_path={train_config_json}"]
else:
    print("No existing run — starting fresh")

# ── Policy config ────────────────────────────────────────────────────────
policy_cfg = ACTConfig(
    input_features  = input_features,
    output_features = output_features,
    normalization_mapping = {
        "VISUAL" : NormalizationMode.MEAN_STD,
        "STATE"  : NormalizationMode.MEAN_STD,
        "ACTION" : NormalizationMode.MEAN_STD,
    },
    chunk_size      = CHUNK_SIZE,
    n_action_steps  = CHUNK_SIZE,
    n_obs_steps     = 1,
    dim_model       = 256,
    n_heads         = 8,
    dim_feedforward = 3200,
    n_encoder_layers= 4,
    n_decoder_layers= 1,
    use_vae         = True,
    latent_dim      = 32,
    kl_weight       = 10.0,
)

# ── Dataset config ───────────────────────────────────────────────────────
dataset_cfg = DatasetConfig(
    repo_id  = DATASET_REPO_ID,
    episodes = None,
)

# ── Top-level training config ────────────────────────────────────────────
train_cfg = TrainPipelineConfig(
    dataset        = dataset_cfg,
    policy         = policy_cfg,
    output_dir     = Path(CHECKPOINT_DIR),
    resume         = IS_RESUME,
    steps          = STEPS,
    batch_size     = BATCH_SIZE,
    num_workers    = 4,
    eval_freq      = -1,
    log_freq       = 200,
    save_checkpoint= True,
    save_freq      = SAVE_FREQ,
    seed           = 42,
)
train_cfg.policy.push_to_hub = False

# Optionally enable W&B
if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name=f"{DATASET_NAME}_act", resume="allow")
    train_cfg.wandb.enable  = True
    train_cfg.wandb.project = WANDB_PROJECT

# ── Launch training ──────────────────────────────────────────────────────
print(f"\nStarting training ({STEPS:,} steps, saving every {SAVE_FREQ:,})...")
train(train_cfg)

---
## Step 8 — Resume after disconnection

If your Colab session disconnected or you closed the browser:

1. Open this notebook again
2. Run **Steps 1 → 6** in order (GPU check, Drive mount, install, config, login, dataset preview)
3. Run the cell below **instead of** Step 7

It loads the latest checkpoint from Drive and continues from where it left off.

In [ ]:
# ── Explicit resume cell ─────────────────────────────────────────────────
# Run this cell (instead of Step 7) when resuming after a disconnection.

import sys
from pathlib import Path

import lerobot.datasets.feature_utils as _fu
import lerobot.policies.factory as _pf

_orig_d2pf = _fu.dataset_to_policy_features

def _d2pf_no_names(features):
    fixed = {}
    for k, v in features.items():
        v = dict(v)
        if len(v.get("shape", [])) == 3 and "names" not in v:
            v["names"] = ["channel", "height", "width"]
        fixed[k] = v
    return _orig_d2pf(fixed)

_fu.dataset_to_policy_features = _d2pf_no_names
_pf.dataset_to_policy_features = _d2pf_no_names

from lerobot.configs.default import DatasetConfig
from lerobot.configs.train import TrainPipelineConfig
from lerobot.configs.types import NormalizationMode
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.scripts.lerobot_train import train

ckpt_root         = Path(CHECKPOINT_DIR)
train_config_json = ckpt_root / "train_config.json"

if not train_config_json.is_file():
    print("No train_config.json found in", CHECKPOINT_DIR)
    print("Run Step 7 to start a fresh training run.")
else:
    checkpoints = sorted(ckpt_root.glob("checkpoints/*/"))
    if checkpoints:
        print(f"Found {len(checkpoints)} checkpoint(s). Resuming from: {checkpoints[-1].name}")
    else:
        print("Config found but no checkpoint saved yet — will resume from step 0")

    sys.argv = [sys.argv[0], f"--config_path={train_config_json}"]

    policy_cfg = ACTConfig(
        input_features   = input_features,
        output_features  = output_features,
        normalization_mapping = {
            "VISUAL" : NormalizationMode.MEAN_STD,
            "STATE"  : NormalizationMode.MEAN_STD,
            "ACTION" : NormalizationMode.MEAN_STD,
        },
        chunk_size       = CHUNK_SIZE,
        n_action_steps   = CHUNK_SIZE,
        n_obs_steps      = 1,
        dim_model        = 256,
        n_heads          = 8,
        dim_feedforward  = 3200,
        n_encoder_layers = 4,
        n_decoder_layers = 1,
        use_vae          = True,
        latent_dim       = 32,
        kl_weight        = 10.0,
    )

    train_cfg = TrainPipelineConfig(
        dataset        = DatasetConfig(repo_id=DATASET_REPO_ID),
        policy         = policy_cfg,
        output_dir     = ckpt_root,
        resume         = True,
        steps          = STEPS,
        batch_size     = BATCH_SIZE,
        num_workers    = 4,
        eval_freq      = -1,
        log_freq       = 200,
        save_checkpoint= True,
        save_freq      = SAVE_FREQ,
        seed           = 42,
    )
    train_cfg.policy.push_to_hub = False

    train(train_cfg)

---
## Step 9 — Inspect checkpoints

Lists all saved checkpoints and their sizes.

In [ ]:
from pathlib import Path

ckpt_root = Path(CHECKPOINT_DIR) / "checkpoints"

if not ckpt_root.exists():
    print("No checkpoints yet — run Step 7 first.")
else:
    ckpts = sorted(ckpt_root.iterdir())
    print(f"{'Checkpoint':<30} {'Size':>10}")
    print("-" * 42)
    total = 0
    for c in ckpts:
        size = sum(f.stat().st_size for f in c.rglob("*") if f.is_file())
        total += size
        print(f"{c.name:<30} {size/1e6:>9.1f} MB")
    print("-" * 42)
    print(f"{'Total:':<30} {total/1e6:>9.1f} MB  ({len(ckpts)} checkpoints)")
    print(f"\nDrive path: {ckpt_root}")

---
## Troubleshooting

**Out of memory (CUDA OOM)**  
Reduce `BATCH_SIZE` to 4 in Step 4, then re-run Steps 6–7.

**`eval_freq` error / environment not found**  
Make sure `eval_freq = -1` in the training config. Evaluation requires a simulation environment which is not available in Colab.

**`KeyError` on a feature name**  
Step 6 must complete before Step 7 — it builds `input_features` and `output_features` from your dataset. If the kernel restarted, re-run Steps 1–6.

**Training loss not decreasing after 20k steps**  
- Try lowering `CHUNK_SIZE` to 25 (shorter prediction horizon)
- Increase `BATCH_SIZE` to 16 if VRAM allows
- Collect more episodes (aim for 100+)

**W&B not logging**  
Run `!wandb login` in a new cell and paste your API key from https://wandb.ai/settings